# 正则化（Regularization）— PyTorch 版

> 本 Notebook 是 TensorFlow 正则化系列的 PyTorch 等价实现，将以下 4 个 TF Notebook 合并为一：
>
> - [L2正则化.ipynb](./L2正则化.ipynb) — L2 / Ridge / Weight Decay
> - [01-L1正则化.ipynb](./01-L1正则化.ipynb) — L1 / Lasso
> - [L1和L2正则化.ipynb](./L1和L2正则化.ipynb) — Elastic Net（L1+L2）
> - [02-正则化用于所有层.ipynb](./02-正则化用于所有层.ipynb) — 批量应用正则化

## 内容概览

| 章节 | 主题 | PyTorch 实现方式 |
|------|------|------------------|
| 1 | L2 正则化（Weight Decay） | `optimizer(weight_decay=...)` 或手动添加 |
| 2 | L1 正则化（Lasso） | 手动实现：`torch.abs(param).sum()` |
| 3 | Elastic Net（L1+L2） | 手动组合 L1 + weight_decay |
| 4 | 正则化应用于所有层 | 工厂函数 / 手动遍历参数 |
| 5 | TF vs PyTorch 对照 | API 映射与差异 |
| 6 | 练习 | 动手实践

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# 设备选择 / Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 版本: {torch.__version__}")
print(f"设备: {device}")

## 数据准备：California Housing

使用 California Housing 数据集（回归任务），便于观察正则化对过拟合的影响。

In [ ]:
# 加载 California Housing 数据集 / Load California Housing dataset
housing = fetch_california_housing()
X, y = housing.data, housing.target

# 划分训练/验证/测试集 / Split into train/val/test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

# 标准化 / Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 转为 PyTorch 张量 / Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1).to(device)

# DataLoader / 数据加载器
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"训练集: {X_train.shape}")
print(f"验证集: {X_val.shape}")
print(f"测试集: {X_test.shape}")
print(f"特征数: {X_train.shape[1]}")

## 通用训练与可视化工具

定义通用的训练函数和绘图函数，供后续各章节复用。

In [ ]:
def train_model(model, train_loader, X_val, y_val, epochs=100, lr=1e-3,
                weight_decay=0.0, l1_lambda=0.0, verbose=True):
    """
    通用训练函数 / General training function.

    Parameters
    ----------
    model : nn.Module
        PyTorch 模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    X_val, y_val : Tensor
        验证数据 / Validation data
    epochs : int
        训练轮数 / Number of training epochs
    lr : float
        学习率 / Learning rate
    weight_decay : float
        L2 正则化系数（传入优化器）/ L2 reg coefficient (passed to optimizer)
    l1_lambda : float
        L1 正则化系数（手动添加）/ L1 reg coefficient (manually added)
    verbose : bool
        是否打印进度 / Whether to print progress

    Returns
    -------
    dict : {'train_loss', 'val_loss', 'train_mae', 'val_mae'}
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_mae': [], 'val_mae': []
    }

    for epoch in range(epochs):
        # 训练阶段 / Training phase
        model.train()
        epoch_loss = 0.0
        epoch_mae = 0.0
        n_batches = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # 手动添加 L1 正则化 / Manually add L1 regularization
            if l1_lambda > 0:
                l1_penalty = sum(
                    torch.abs(param).sum() for param in model.parameters()
                )
                loss = loss + l1_lambda * l1_penalty

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_mae += torch.abs(y_pred - y_batch).mean().item()
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        avg_train_mae = epoch_mae / n_batches

        # 验证阶段 / Validation phase
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val).item()
            val_mae = torch.abs(y_val_pred - y_val).mean().item()

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(val_mae)

        if verbose and (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val MAE: {val_mae:.4f}")

    return history


def plot_history(histories, title='训练曲线对比', metric='loss'):
    """
    绘制训练曲线对比图 / Plot training curve comparison.

    Parameters
    ----------
    histories : dict
        {label: history_dict} 键值对 / {label: history_dict} pairs
    title : str
        图表标题 / Chart title
    metric : str
        'loss' 或 'mae' / 'loss' or 'mae'
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown']

    for idx, (label, hist) in enumerate(histories.items()):
        c = colors[idx % len(colors)]
        key = metric
        axes[0].plot(hist[f'train_{key}'], c=c, label=f'{label} (训练)')
        axes[0].plot(hist[f'val_{key}'], c=c, linestyle='--', label=f'{label} (验证)')
        axes[1].plot(hist[f'train_{key}'], c=c, label=f'{label} (训练)')
        axes[1].plot(hist[f'val_{key}'], c=c, linestyle='--', label=f'{label} (验证)')

    axes[0].set_title(f'{title} - 训练 vs 验证')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel(metric.upper())
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # 仅验证曲线对比 / Validation curves only
    for idx, (label, hist) in enumerate(histories.items()):
        c = colors[idx % len(colors)]
        axes[1].plot(hist[f'val_{key}'], c=c, label=label)
    axes[1].set_title(f'{title} - 验证曲线对比')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel(f'Val {metric.upper()}')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_weight_distribution(models_dict, layer_idx=0):
    """
    可视化模型权重分布对比 / Visualize and compare weight distributions.

    Parameters
    ----------
    models_dict : dict
        {label: model} 键值对 / {label: model} pairs
    layer_idx : int
        要可视化的层索引 / Layer index to visualize
    """
    n = len(models_dict)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for idx, (label, model) in enumerate(models_dict.items()):
        # 获取第 layer_idx 个 Linear 层的权重 / Get weights of the layer_idx-th Linear layer
        linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
        weights = linear_layers[layer_idx].weight.data.cpu().numpy().flatten()
        axes[idx].hist(weights, bins=50, density=True, alpha=0.7,
                       color=['blue', 'red', 'green', 'purple'][idx % 4])
        axes[idx].set_title(f'{label}\n均值={weights.mean():.4f}, '
                            f'标准差={weights.std():.4f}')
        axes[idx].set_xlabel('权重值 / Weight')
        axes[idx].set_ylabel('密度 / Density')
        axes[idx].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # 打印权重范数 / Print weight norms
    print('权重范数对比 / Weight norm comparison:')
    for label, model in models_dict.items():
        linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
        weights = linear_layers[layer_idx].weight.data.cpu().numpy().flatten()
        print(f"  {label}: L2 范数 = {np.linalg.norm(weights):.4f}")


print("工具函数定义完成 / Utility functions defined.")

---

## 1. L2 正则化（Ridge / Weight Decay）

### 核心思想

L2 正则化通过在损失函数中添加权重平方和的惩罚项，限制权重的大小，防止过拟合。

### 数学表达

```
Loss_total = Loss_original + λ × Σwᵢ²
```

### PyTorch 两种实现方式

| 方式 | 代码 | 优点 | 缺点 |
|------|------|------|------|
| **方式一**：优化器 weight_decay | `optim.Adam(..., weight_decay=0.01)` | 简洁，一行搞定 | 不够灵活，对 L1 无效 |
| **方式二**：手动添加正则项 | `loss + lambda * sum(p.pow(2).sum() for p in params)` | 灵活可控 | 代码稍多 |

In [ ]:
class RegressionModel(nn.Module):
    """
    基础回归模型 / Basic regression model.
    用于 California Housing 预测 / Used for California Housing prediction.
    """
    def __init__(self, input_dim=8, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 128, 64]
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ELU())
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

        # He 初始化 / He initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)

print("RegressionModel 定义完成 / RegressionModel defined.")

### 1.1 方式一：使用 optimizer 的 weight_decay 参数

这是最简单的方式，PyTorch 优化器内置了 `weight_decay` 参数，等价于 L2 正则化。

In [ ]:
# 无正则化模型 / No regularization model
model_no_reg = RegressionModel()
hist_no_reg = train_model(
    model_no_reg, train_loader, X_val_t, y_val_t,
    epochs=100, lr=1e-3, weight_decay=0.0
)

# L2 正则化模型（weight_decay） / L2 regularized model (weight_decay)
model_l2_wd = RegressionModel()
hist_l2_wd = train_model(
    model_l2_wd, train_loader, X_val_t, y_val_t,
    epochs=100, lr=1e-3, weight_decay=0.01
)

print("\n--- 方式一：weight_decay ---")
print(f"无正则化 - 最终验证 MAE: {hist_no_reg['val_mae'][-1]:.4f}")
print(f"L2 (wd=0.01) - 最终验证 MAE: {hist_l2_wd['val_mae'][-1]:.4f}")

### 1.2 方式二：手动添加 L2 正则项

手动实现 L2 正则化，更灵活，可以对不同层使用不同系数。

In [ ]:
def train_model_manual_l2(model, train_loader, X_val, y_val,
                          epochs=100, lr=1e-3, l2_lambda=0.01, verbose=True):
    """
    手动 L2 正则化训练函数 / Training function with manual L2 regularization.

    手动计算 L2 惩罚项并加到损失上，
    而不使用优化器的 weight_decay 参数。
    This manually computes the L2 penalty and adds it to the loss,
    instead of using the optimizer's weight_decay parameter.
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    # 注意：weight_decay=0，不使用优化器自带的 L2 / Note: weight_decay=0
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_mae': [], 'val_mae': []
    }

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        epoch_mae = 0.0
        n_batches = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # 手动添加 L2 正则化 / Manually add L2 regularization
            l2_penalty = sum(
                param.pow(2).sum() for param in model.parameters()
            )
            loss = loss + l2_lambda * l2_penalty

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_mae += torch.abs(y_pred - y_batch).mean().item()
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        avg_train_mae = epoch_mae / n_batches

        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val).item()
            val_mae = torch.abs(y_val_pred - y_val).mean().item()

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(val_mae)

        if verbose and (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val MAE: {val_mae:.4f}")

    return history


# 训练手动 L2 正则化模型 / Train with manual L2 regularization
model_l2_manual = RegressionModel()
hist_l2_manual = train_model_manual_l2(
    model_l2_manual, train_loader, X_val_t, y_val_t,
    epochs=100, lr=1e-3, l2_lambda=0.01
)

print("\n--- 方式二：手动 L2 ---")
print(f"L2 (manual, λ=0.01) - 最终验证 MAE: {hist_l2_manual['val_mae'][-1]:.4f}")

### 1.3 两种方式对比

In [ ]:
# 对比三种模型的训练曲线 / Compare training curves of three models
plot_history({
    '无正则化': hist_no_reg,
    'L2 (weight_decay)': hist_l2_wd,
    'L2 (手动)': hist_l2_manual
}, title='L2 正则化对比 - 损失曲线', metric='loss')

# 对比 MAE 曲线 / Compare MAE curves
plot_history({
    '无正则化': hist_no_reg,
    'L2 (weight_decay)': hist_l2_wd,
    'L2 (手动)': hist_l2_manual
}, title='L2 正则化对比 - MAE 曲线', metric='mae')

In [ ]:
# 权重分布对比 / Compare weight distributions
plot_weight_distribution({
    '无正则化': model_no_reg,
    'L2 (weight_decay)': model_l2_wd,
    'L2 (手动)': model_l2_manual
}, layer_idx=0)

# 测试集评估 / Evaluate on test set
print('\n测试集评估 / Test set evaluation:')
for name, mdl in [('无正则化', model_no_reg),
                   ('L2 (weight_decay)', model_l2_wd),
                   ('L2 (手动)', model_l2_manual)]:
    mdl.eval()
    with torch.no_grad():
        test_pred = mdl(X_test_t)
        test_mae = torch.abs(test_pred - y_test_t).mean().item()
    print(f"  {name}: Test MAE = {test_mae:.4f}")

---

## 2. L1 正则化（Lasso）

### 核心思想

L1 正则化通过在损失函数中添加权重绝对值之和的惩罚项，促使模型学习稀疏特征。

### 数学表达

```
Loss_total = Loss_original + λ × Σ|wᵢ|
```

### 关键区别

> **PyTorch 优化器没有内置 L1 正则化！** 必须手动实现。
>
> TensorFlow/Keras 有 `keras.regularizers.l1()`，但 PyTorch 没有等价的优化器参数。

| 特性 | L1 正则化 | L2 正则化 |
|------|-----------|----------|
| 惩罚项 | Σ|wᵢ| | Σwᵢ² |
| 权重分布 | 稀疏（多个0） | 小而分散 |
| 特征选择 | 是 | 否 |
| 计算效率 | 较低 | 较高 |
| PyTorch 内置 | 无 | weight_decay |

In [ ]:
# L1 正则化训练 / L1 regularization training
# 使用之前定义的 train_model 函数，通过 l1_lambda 参数实现

model_l1 = RegressionModel()
hist_l1 = train_model(
    model_l1, train_loader, X_val_t, y_val_t,
    epochs=100, lr=1e-3, weight_decay=0.0, l1_lambda=0.001
)

print(f"\nL1 (λ=0.001) - 最终验证 MAE: {hist_l1['val_mae'][-1]:.4f}")

### 2.1 L1 正则化的稀疏性效果

In [ ]:
# 对比 L1 正则化与无正则化的训练曲线 / Compare L1 vs no regularization
plot_history({
    '无正则化': hist_no_reg,
    'L1 (λ=0.001)': hist_l1
}, title='L1 正则化对比', metric='loss')

# 权重分布对比 / Compare weight distributions
plot_weight_distribution({
    '无正则化': model_no_reg,
    'L1 (λ=0.001)': model_l1
}, layer_idx=0)

In [ ]:
# 统计接近零的权重比例 / Calculate sparsity ratio
threshold = 0.01

def count_sparsity(model, thresh=0.01):
    """
    计算模型中接近零的权重比例 / Calculate the proportion of near-zero weights.
    """
    total = 0
    sparse = 0
    for param in model.parameters():
        total += param.numel()
        sparse += (torch.abs(param) < thresh).sum().item()
    return sparse / total

sparse_no_reg = count_sparsity(model_no_reg, threshold)
sparse_l1 = count_sparsity(model_l1, threshold)

print(f"接近零的权重比例 (阈值={threshold}) / Near-zero weight ratio (threshold={threshold}):")
print(f"  无正则化 / No reg:  {sparse_no_reg:.2%}")
print(f"  L1 正则化 / L1 reg: {sparse_l1:.2%}")
print("\nL1 正则化使更多权重趋近于零 → 稀疏性效果 / L1 drives more weights to zero → sparsity")

### 2.2 L1 正则化强度对比

In [ ]:
# 不同 L1 强度的对比 / Compare different L1 strengths
l1_values = [0.0, 0.0001, 0.001, 0.01]
l1_histories = {}

for l1_val in l1_values:
    label = f'L1 λ={l1_val}' if l1_val > 0 else '无正则化'
    print(f"训练 {label}...")
    m = RegressionModel()
    h = train_model(
        m, train_loader, X_val_t, y_val_t,
        epochs=100, lr=1e-3, weight_decay=0.0, l1_lambda=l1_val, verbose=False
    )
    l1_histories[label] = h
    print(f"  验证 MAE: {h['val_mae'][-1]:.4f}, 稀疏率: {count_sparsity(m, 0.01):.2%}")

plot_history(l1_histories, title='L1 正则化强度对比', metric='loss')

print("\nL1 正则化强度选择建议 / L1 strength selection guide:")
print("=" * 60)
print(f"{'λ 值':<12} {'效果':<40}")
print("-" * 60)
print(f"{'0.0001':<12} {'轻微正则化，适合已经不太过拟合的模型':<40}")
print(f"{'0.001':<12} {'中等正则化，常用起始值':<40}")
print(f"{'0.01':<12} {'较强正则化，明显稀疏效果':<40}")
print(f"{'0.1':<12} {'强正则化，可能导致欠拟合':<40}")

---

## 3. Elastic Net 正则化（L1 + L2）

### 核心思想

Elastic Net 结合了 L1 和 L2 正则化的优点，同时获得稀疏性和权重约束。

### 数学表达

```
Loss_total = Loss_original + λ₁ × Σ|wᵢ| + λ₂ × Σwᵢ²
```

### 优势

| 特性 | L1 | L2 | Elastic Net |
|------|----|----|-------------|
| 稀疏性 | 强 | 无 | 中等 |
| 权重约束 | 弱 | 强 | 强 |
| 相关特征处理 | 随机选一个 | 均匀分配 | 分组选择 |

### PyTorch 实现

组合使用 `weight_decay`（L2）和手动 L1 正则项：

In [ ]:
# Elastic Net: L1 (手动) + L2 (weight_decay) / Elastic Net: L1 (manual) + L2 (weight_decay)
model_elastic = RegressionModel()
hist_elastic = train_model(
    model_elastic, train_loader, X_val_t, y_val_t,
    epochs=100, lr=1e-3,
    weight_decay=0.001,   # L2 部分 / L2 part
    l1_lambda=0.001       # L1 部分 / L1 part
)

print(f"\nElastic Net (L1=0.001, L2=0.001) - 最终验证 MAE: {hist_elastic['val_mae'][-1]:.4f}")

### 3.1 三种正则化全面对比

In [ ]:
# 四种模型全面对比 / Full comparison of four models
plot_history({
    '无正则化': hist_no_reg,
    'L2 (wd=0.01)': hist_l2_wd,
    'L1 (λ=0.001)': hist_l1,
    'Elastic Net': hist_elastic
}, title='正则化方法全面对比', metric='loss')

plot_history({
    '无正则化': hist_no_reg,
    'L2 (wd=0.01)': hist_l2_wd,
    'L1 (λ=0.001)': hist_l1,
    'Elastic Net': hist_elastic
}, title='正则化方法全面对比', metric='mae')

In [ ]:
# 权重分布对比 / Weight distribution comparison
plot_weight_distribution({
    '无正则化': model_no_reg,
    'L2 (wd=0.01)': model_l2_wd,
    'L1 (λ=0.001)': model_l1,
    'Elastic Net': model_elastic
}, layer_idx=0)

# 稀疏性统计 / Sparsity statistics
print('\n稀疏性统计 / Sparsity statistics (threshold=0.01):')
for name, mdl in [('无正则化', model_no_reg),
                   ('L2 (wd=0.01)', model_l2_wd),
                   ('L1 (λ=0.001)', model_l1),
                   ('Elastic Net', model_elastic)]:
    print(f"  {name}: {count_sparsity(mdl, 0.01):.2%}")

# 测试集评估 / Test set evaluation
print('\n测试集评估 / Test set evaluation:')
for name, mdl in [('无正则化', model_no_reg),
                   ('L2 (wd=0.01)', model_l2_wd),
                   ('L1 (λ=0.001)', model_l1),
                   ('Elastic Net', model_elastic)]:
    mdl.eval()
    with torch.no_grad():
        test_pred = mdl(X_test_t)
        test_mae = torch.abs(test_pred - y_test_t).mean().item()
    print(f"  {name}: Test MAE = {test_mae:.4f}")

### 3.2 选择建议

| 场景 | 推荐正则化 |
|------|------------|
| 需要特征选择 | L1 |
| 防止过拟合 | L2 |
| 特征高度相关 | Elastic Net |
| 不确定时 | 先尝试 L2 |

---

## 4. 正则化应用于所有层

### 问题背景

当需要对多个层应用相同的正则化配置时，逐层手写正则项会变得冗长且难以维护。

### 解决方案

在 PyTorch 中，我们可以通过以下方式批量应用正则化：

1. **工厂函数模式** — 类似 TF 的 `functools.partial`，创建预配置的层
2. **自定义模型类** — 在模型内部统一管理正则化
3. **参数遍历** — 在训练循环中统一处理所有参数

In [ ]:
# 方式一：工厂函数模式 / Approach 1: Factory function pattern
# 类似 TF 中 functools.partial 的用法

def make_regularized_linear(in_features, out_features, l2_wd=0.001):
    """
    创建带 L2 正则化信息的线性层 / Create a Linear layer with L2 regularization info.

    注意：PyTorch 中正则化不在层级别设置，
    而是通过 weight_decay 或手动计算。
    Note: In PyTorch, regularization is not set at the layer level,
    but via weight_decay or manual computation.

    Parameters
    ----------
    in_features : int
        输入特征数 / Number of input features
    out_features : int
        输出特征数 / Number of output features
    l2_wd : float
        L2 正则化系数 / L2 regularization coefficient (for reference)
    """
    layer = nn.Linear(in_features, out_features)
    nn.init.kaiming_normal_(layer.weight, nonlinearity='linear')
    nn.init.zeros_(layer.bias)
    # 存储正则化信息供后续参考 / Store reg info for later reference
    layer.l2_weight_decay = l2_wd
    return layer


# 方式二：自定义模型类，内置正则化配置 / Approach 2: Custom model class
class RegularizedRegressionModel(nn.Module):
    """
    带统一正则化配置的回归模型 / Regression model with unified regularization config.

    所有隐藏层自动应用相同的正则化策略。
    All hidden layers automatically apply the same regularization strategy.
    """
    def __init__(self, input_dim=8, hidden_dims=None,
                 activation='elu', l2_wd=0.001, l1_lambda=0.0):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 128, 64]
        self.l2_wd = l2_wd
        self.l1_lambda = l1_lambda

        # 选择激活函数 / Choose activation
        act_map = {
            'elu': nn.ELU,
            'relu': nn.ReLU,
            'selu': nn.SELU,
        }
        act_cls = act_map.get(activation, nn.ELU)

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(make_regularized_linear(prev_dim, h_dim, l2_wd=l2_wd))
            layers.append(act_cls())
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    def get_l1_penalty(self):
        """
        计算 L1 惩罚项 / Compute L1 penalty.
        """
        return sum(torch.abs(p).sum() for p in self.parameters())

    def get_l2_penalty(self):
        """
        计算 L2 惩罚项 / Compute L2 penalty.
        """
        return sum(p.pow(2).sum() for p in self.parameters())


# 创建模型 / Create models
model_all_l2 = RegularizedRegressionModel(l2_wd=0.001, l1_lambda=0.0)
model_all_elastic = RegularizedRegressionModel(l2_wd=0.001, l1_lambda=0.001)

print("正则化模型创建完成 / Regularized models created.")
print(f"model_all_l2: L2={model_all_l2.l2_wd}, L1={model_all_l2.l1_lambda}")
print(f"model_all_elastic: L2={model_all_elastic.l2_wd}, L1={model_all_elastic.l1_lambda}")

In [ ]:
# 方式三：参数遍历 — 按层类型区分正则化 / Approach 3: Parameter traversal
# 可以对不同类型的参数应用不同的正则化策略

def compute_layerwise_regularization(model, l2_lambda=0.001, l1_lambda=0.0,
                                     skip_bias=True):
    """
    按层计算正则化损失 / Compute per-layer regularization loss.

    可以选择跳过偏置项（通常不对偏置做正则化）。
    Can optionally skip bias terms (biases are usually not regularized).

    Parameters
    ----------
    model : nn.Module
        PyTorch 模型 / PyTorch model
    l2_lambda : float
        L2 正则化系数 / L2 regularization coefficient
    l1_lambda : float
        L1 正则化系数 / L1 regularization coefficient
    skip_bias : bool
        是否跳过偏置项 / Whether to skip bias terms

    Returns
    -------
    torch.Tensor : 正则化损失 / Regularization loss
    """
    reg_loss = torch.tensor(0.0, device=device)
    for name, param in model.named_parameters():
        if skip_bias and 'bias' in name:
            continue  # 跳过偏置 / Skip bias
        if l2_lambda > 0:
            reg_loss = reg_loss + l2_lambda * param.pow(2).sum()
        if l1_lambda > 0:
            reg_loss = reg_loss + l1_lambda * torch.abs(param).sum()
    return reg_loss


# 演示：遍历模型参数 / Demo: iterate over model parameters
print("模型参数结构 / Model parameter structure:")
for name, param in model_all_l2.named_parameters():
    is_bias = 'bias' in name
    print(f"  {name}: shape={param.shape}, "
          f"{'偏置(通常不正则化)' if is_bias else '权重(正则化)'}")

In [ ]:
# 使用 RegularizedRegressionModel 训练 / Train using RegularizedRegressionModel
def train_regularized_model(model, train_loader, X_val, y_val,
                             epochs=100, lr=1e-3, verbose=True):
    """
    使用模型自带的正则化配置进行训练 / Train with model's built-in regularization config.
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    # 使用模型配置的 weight_decay / Use model's configured weight_decay
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=model.l2_wd)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_mae': [], 'val_mae': []
    }

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        epoch_mae = 0.0
        n_batches = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # L2 通过 weight_decay 自动处理 / L2 handled by weight_decay
            # L1 需要手动添加 / L1 needs manual addition
            if model.l1_lambda > 0:
                loss = loss + model.l1_lambda * model.get_l1_penalty()

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_mae += torch.abs(y_pred - y_batch).mean().item()
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        avg_train_mae = epoch_mae / n_batches

        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val).item()
            val_mae = torch.abs(y_val_pred - y_val).mean().item()

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(val_mae)

        if verbose and (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val MAE: {val_mae:.4f}")

    return history


# 训练 / Train
print("训练 L2 全层正则化模型...")
hist_all_l2 = train_regularized_model(
    model_all_l2, train_loader, X_val_t, y_val_t, epochs=100
)

print("\n训练 Elastic Net 全层正则化模型...")
hist_all_elastic = train_regularized_model(
    model_all_elastic, train_loader, X_val_t, y_val_t, epochs=100
)

print(f"\n全层 L2 (wd=0.001) - 最终验证 MAE: {hist_all_l2['val_mae'][-1]:.4f}")
print(f"全层 Elastic Net (L1=0.001, L2=0.001) - 最终验证 MAE: {hist_all_elastic['val_mae'][-1]:.4f}")

In [ ]:
# 对比全层正则化效果 / Compare all-layer regularization effects
plot_history({
    '无正则化': hist_no_reg,
    '全层 L2': hist_all_l2,
    '全层 Elastic Net': hist_all_elastic
}, title='全层正则化对比', metric='loss')

# 权重分布对比 / Weight distribution comparison
plot_weight_distribution({
    '无正则化': model_no_reg,
    '全层 L2': model_all_l2,
    '全层 Elastic Net': model_all_elastic
}, layer_idx=0)

---

## 5. TF vs PyTorch 对照

### API 映射表

| 功能 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| **L2 正则化** | `kernel_regularizer=keras.regularizers.l2(0.01)` | `optim.Adam(..., weight_decay=0.01)` 或手动添加 `λ * Σw²` |
| **L1 正则化** | `kernel_regularizer=keras.regularizers.l1(0.01)` | 手动：`λ * Σ|w|` （无内置） |
| **Elastic Net** | `kernel_regularizer=keras.regularizers.l1_l2(l1=0.01, l2=0.01)` | `weight_decay` (L2) + 手动 L1 |
| **Activity 正则化** | `activity_regularizer=keras.regularizers.l2(0.01)` | 手动：`λ * Σ|output|²` |
| **批量应用** | `functools.partial(keras.layers.Dense, kernel_regularizer=...)` | 自定义模型类 / 工厂函数 / 参数遍历 |
| **按层不同系数** | 各层独立设置 `kernel_regularizer` | 参数遍历时按 `name` 过滤 |

### 关键差异

1. **正则化位置不同**
   - **TF/Keras**：正则化是**层级别**的，每个层可以独立设置 `kernel_regularizer`
   - **PyTorch**：正则化是**优化器级别**的（weight_decay）或**手动计算**的

2. **weight_decay vs 真正的 L2**
   - 在 **SGD** 中，`weight_decay` 等价于在损失中添加 L2 项
   - 在 **Adam** 中，`weight_decay` 的实现方式可能不同（建议使用 `AdamW` 获得解耦的 weight_decay）

3. **L1 正则化**
   - **TF/Keras**：有内置 `keras.regularizers.l1()`
   - **PyTorch**：没有内置，必须手动实现

4. **偏置项**
   - **TF/Keras**：`kernel_regularizer` 只作用于权重，不影响偏置
   - **PyTorch**：`weight_decay` 默认同时作用于权重和偏置（需要注意！）
   - **建议**：手动正则化时跳过偏置项

In [ ]:
# 演示：AdamW vs Adam + weight_decay 的区别 / Demo: AdamW vs Adam + weight_decay
# AdamW 将 weight_decay 从梯度更新中解耦，效果更稳定

model_adamw = RegressionModel()
model_adamw = model_adamw.to(device)

# 使用 AdamW / Using AdamW
optimizer_adamw = optim.AdamW(model_adamw.parameters(), lr=1e-3, weight_decay=0.01)

print("AdamW vs Adam + weight_decay:")
print("  Adam + weight_decay: weight_decay 被加入梯度的一阶矩估计")
print("  AdamW + weight_decay: weight_decay 与梯度更新解耦，效果更接近真正的 L2")
print("  推荐在需要 L2 正则化时优先使用 AdamW")
print("\nRecommendation: Prefer AdamW when using L2 regularization.")

In [ ]:
# 演示：跳过偏置项的正则化 / Demo: Skip bias in regularization
# 这是 PyTorch 中一个重要的实践 / This is an important practice in PyTorch

def train_skip_bias_l2(model, train_loader, X_val, y_val,
                       epochs=100, lr=1e-3, l2_lambda=0.01, verbose=True):
    """
    训练时仅对权重做 L2 正则化，跳过偏置 / L2 regularization on weights only, skip bias.

    这与 TF/Keras 的 kernel_regularizer 行为一致。
    This matches TF/Keras kernel_regularizer behavior.
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    # weight_decay=0, 手动控制 / weight_decay=0, manual control
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)

    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        epoch_mae = 0.0
        n_batches = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # 仅对权重（非偏置）做 L2 / L2 on weights only (skip bias)
            l2_penalty = sum(
                p.pow(2).sum()
                for name, p in model.named_parameters()
                if 'weight' in name  # 跳过偏置 / Skip bias
            )
            loss = loss + l2_lambda * l2_penalty

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_mae += torch.abs(y_pred - y_batch).mean().item()
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        avg_train_mae = epoch_mae / n_batches

        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val).item()
            val_mae = torch.abs(y_val_pred - y_val).mean().item()

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(val_mae)

        if verbose and (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val MAE: {val_mae:.4f}")

    return history

# 训练跳过偏置的 L2 模型 / Train with bias-skipped L2
model_skip_bias = RegressionModel()
hist_skip_bias = train_skip_bias_l2(
    model_skip_bias, train_loader, X_val_t, y_val_t,
    epochs=100, lr=1e-3, l2_lambda=0.01
)

print(f"\nL2 (跳过偏置, λ=0.01) - 最终验证 MAE: {hist_skip_bias['val_mae'][-1]:.4f}")
print("\n注意：跳过偏置的正则化与 TF/Keras 的 kernel_regularizer 行为更一致")
print("Note: Skipping bias in regularization better matches TF/Keras kernel_regularizer behavior.")

In [ ]:
# 完整总结对比 / Full summary comparison
print('=' * 70)
print('所有模型测试集评估 / All models test set evaluation:')
print('=' * 70)

all_models = [
    ('无正则化', model_no_reg),
    ('L2 (weight_decay=0.01)', model_l2_wd),
    ('L2 (手动, λ=0.01)', model_l2_manual),
    ('L2 (跳过偏置, λ=0.01)', model_skip_bias),
    ('L1 (λ=0.001)', model_l1),
    ('Elastic Net (L1=0.001, L2=0.001)', model_elastic),
    ('全层 L2 (wd=0.001)', model_all_l2),
    ('全层 Elastic Net (L1=0.001, L2=0.001)', model_all_elastic),
]

for name, mdl in all_models:
    mdl.eval()
    with torch.no_grad():
        test_pred = mdl(X_test_t)
        test_mae = torch.abs(test_pred - y_test_t).mean().item()
        test_rmse = torch.sqrt(nn.MSELoss()(test_pred, y_test_t)).item()
    print(f"  {name:<40s} MAE={test_mae:.4f}  RMSE={test_rmse:.4f}")

print('\n' + '=' * 70)
print('关键要点 / Key takeaways:')
print('=' * 70)
print('1. L2 正则化：优化器 weight_decay 或手动添加，推荐 AdamW')
print('2. L1 正则化：PyTorch 无内置，必须手动实现')
print('3. Elastic Net：组合 weight_decay + 手动 L1')
print('4. 偏置项通常不正则化，需手动跳过（PyTorch weight_decay 默认对偏置生效）')
print('5. 自定义模型类 + 参数遍历是批量应用正则化的最佳实践')
print('6. AdamW 比 Adam + weight_decay 更接近真正的 L2 正则化')

---

## 6. 练习

### 练习 1：Activity Regularization

在 TF/Keras 中，`activity_regularizer` 对层的**输出**进行正则化（而非权重）。PyTorch 没有内置此功能。

**任务**：实现一个 PyTorch 等价的 activity regularization，对隐藏层的输出施加 L2 惩罚。

提示：
- 在 `forward()` 中记录隐藏层输出
- 在计算损失时，添加 `λ * Σ(hidden_output²)`
- 对比有无 activity regularization 的效果

```python
# 参考框架 / Reference framework
class ModelWithActivityReg(nn.Module):
    def __init__(self, input_dim=8, activity_lambda=0.01):
        super().__init__()
        self.activity_lambda = activity_lambda
        self.hidden_outputs = []  # 存储隐藏层输出
        # ... 定义层 ...

    def forward(self, x):
        self.hidden_outputs = []
        # ... 前向传播，记录每层输出 ...
        return x

    def get_activity_penalty(self):
        """计算 activity regularization 惩罚项"""
        # TODO: 实现此方法
        pass
```

### 练习 2：不同层使用不同正则化系数

在 TF/Keras 中，每层可以独立设置 `kernel_regularizer`。在 PyTorch 中需要手动实现。

**任务**：修改训练循环，对模型的不同层使用不同的 L2 正则化系数。

要求：
- 第 1 个隐藏层：L2 = 0.01
- 第 2 个隐藏层：L2 = 0.001
- 第 3 个隐藏层：L2 = 0.0001
- 输出层：不正则化

提示：
```python
# 参考框架 / Reference framework
layer_l2_map = {
    'network.0.weight': 0.01,   # 第1层权重
    'network.2.weight': 0.001,  # 第2层权重
    'network.4.weight': 0.0001, # 第3层权重
    # 输出层不在映射中 → 不正则化
}

for name, param in model.named_parameters():
    if name in layer_l2_map:
        l2_reg += layer_l2_map[name] * param.pow(2).sum()
```

### 练习 3：正则化系数网格搜索

**任务**：对 Elastic Net 的 L1 和 L2 系数进行网格搜索，找到最优组合。

要求：
- L1 候选值：[0.0, 0.0001, 0.001, 0.01]
- L2 候选值：[0.0, 0.001, 0.01]
- 绘制热力图展示验证 MAE
- 找到最优 (L1, L2) 组合

提示：
```python
import seaborn as sns  # 可选，用于热力图

l1_candidates = [0.0, 0.0001, 0.001, 0.01]
l2_candidates = [0.0, 0.001, 0.01]
results = np.zeros((len(l1_candidates), len(l2_candidates)))

for i, l1 in enumerate(l1_candidates):
    for j, l2 in enumerate(l2_candidates):
        model = RegressionModel()
        hist = train_model(model, ..., l1_lambda=l1, weight_decay=l2, verbose=False)
        results[i, j] = hist['val_mae'][-1]

# 绘制热力图 / Plot heatmap
plt.figure(figsize=(8, 6))
plt.imshow(results, cmap='viridis_r')
# ... 添加标签 ...
```